## pip install xgboost

In [16]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [17]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

from xgboost import XGBClassifier

In [18]:
df = pd.read_csv("credit_dataset.csv")

X = df[[
    "age",
    "income",
    "debt_to_income",
    "credit_score",
    "loan_amount"
]]

y = df["approved"]

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [20]:
xgb = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [21]:
y_pred_xgb = xgb.predict(X_test)

In [22]:
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
cm_xgb

array([[1049,  151],
       [ 154, 1646]])

In [24]:
from src.evaluate import cost_sensitive_evaluation

results_xgb = cost_sensitive_evaluation(
    y_test,
    y_pred_xgb,
    cost_fp=100,
    cost_fn=10
)

results_xgb

{'confusion_matrix': array([[1049,  151],
        [ 154, 1646]]),
 'false_positives': 151,
 'false_negatives': 154,
 'expected_cost': 16640,
 'cost_per_applicant': 5.546666666666667}

1. ([[1049,  151],
       [ 154, 1646]])
       
2. 16640
3. 5.547
4. reducted
5. Best
6. Can deploy after adding some logic to reduce FP more

In [26]:
y_prob_xgb = xgb.predict_proba(X_test)[:,1]

In [27]:
import numpy as np

thresholds = np.linspace(0.0, 1.0, 101)
costs = []

for t in thresholds:
    y_pred_t = (y_prob_xgb >= t).astype(int)

    results = cost_sensitive_evaluation(
        y_test,
        y_pred_t,
        cost_fp=100,
        cost_fn=10
    )

    costs.append(results["expected_cost"])

In [28]:
optimal_idx = np.argmin(costs)
optimal_threshold = thresholds[optimal_idx]
min_cost = costs[optimal_idx]

optimal_threshold, min_cost

(0.9, 7120)

In [31]:
y_pred_opt = (y_prob_xgb >= optimal_threshold).astype(int)

final_results = cost_sensitive_evaluation(
    y_test,
    y_pred_opt,
    cost_fp=100,
    cost_fn=10
)

final_results

{'confusion_matrix': array([[1188,   12],
        [ 592, 1208]]),
 'false_positives': 12,
 'false_negatives': 592,
 'expected_cost': 7120,
 'cost_per_applicant': 2.3733333333333335}

1. 0.9 is the optimal threshold
2. FP reduced from 157 to 12
3. FN increased from 151 to 592
4. Expected cost reduced from 16640 to 7120 and cost per applicant is 2.37 (less than half of untuned model)
5. This policy is rational because it optimizes for the cost of lending each dollar as opposed to just equally categorising (approving/rejecting) the maximum number of people.